In [ ]:
import gradio as gr
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors  # 👈 确保这里有 AllChem
from rdkit.Chem.Draw import rdMolDraw2D
from PIL import Image
import io
import cairosvg
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
# 1. 加载模型
device="cuda"
model_name = "merged_model"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="auto", 
    dtype=torch.float16)
model.eval()

In [ ]:
SYSTEM_PROMPT = """You are an expert in thermodynamics and vapor-liquid equilibrium.
Your task is to predict equilibrium properties of binary mixtures from molecular structures and operating conditions.
Return only the requested numerical results.
"""
def build_messages(user_input, assistant_output=None):
    user_content = SYSTEM_PROMPT.strip() + "\n\n" + user_input.strip()
    messages = [{"role": "user", "content": user_content}]
    if assistant_output is not None:
        messages.append({"role": "assistant", "content": assistant_output})
    return messages
def parse_vle_output(text, task_type="isobaric"):
    text = str(text)
    y1, T, P = None, None, None
    m = re.search(r"Vapor-phase mole fraction of component 1\s*:\s*([-+]?\d*\.?\d+)", text, re.I)
    if m: y1 = float(m.group(1))
    if task_type == "isobaric":
        m = re.search(r"Equilibrium temperature\s*:\s*([-+]?\d*\.?\d+)", text, re.I)
        if m: T = float(m.group(1))
    else:
        m = re.search(r"Equilibrium pressure\s*:\s*([-+]?\d*\.?\d+)", text, re.I)
        if m: P = float(m.group(1))
    return y1, T, P
def build_isobaric_input(smiles1, smiles2, pressure, x1):
    return f"""Task: Predict the vapor-liquid equilibrium state of a binary mixture under isobaric conditions.
Given:
Component 1 SMILES: {smiles1}
Component 2 SMILES: {smiles2}
Operating pressure: {pressure:.3f} kPa
Liquid-phase mole fraction of component 1: {x1:.3f}
Please predict:
1. Vapor-phase mole fraction of component 1
2. Equilibrium temperature"""
def run_model(user_input):
    messages = build_messages(user_input=user_input, assistant_output=None)
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True
    ).to(device)
    input_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        eos_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.pad_token_id
        outputs = model.generate(
            **inputs, max_new_tokens=64, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_id
        )
    return tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
def draw_phase_diagram(smiles1, smiles2, pressure, n_points=20):
    xs = [round(i / (n_points - 1), 3) for i in range(n_points)]
    x_points, y_points, T_points = [], [], []
    for x in xs:
        try:
            raw = run_model(build_isobaric_input(smiles1, smiles2, pressure, x))
            y1, T, _ = parse_vle_output(raw, "isobaric")
            if T is None:
                continue
            x_points.append(x)
            T_points.append(T)
            y_points.append(y1 if y1 is not None else x)
        except Exception:
            continue
    if not T_points:
        return None
    plt.rcParams.update({
        "font.size": 12,
        "axes.linewidth": 1.2,
        "figure.facecolor": "white",
    })
    fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
    ax.plot(x_points, T_points, "o-", color="#0000FF",
            linewidth=2, markersize=6, label="Bubble point")
    ax.plot(y_points, T_points, "s-", color="#FF0000",
            linewidth=2, markersize=6, label="Dew point")
    #ax.fill_betweenx(T_points, x_points, y_points, color="#2ca02c", alpha=0.12)
    ax.set_xlabel(r"$x_A$", fontsize=25, fontweight="bold")
    ax.set_ylabel("Temperature (K)", fontsize=15, fontweight="bold")
    ax.set_title(f"Predicted  T–x–y Diagram ({pressure:.3f} kPa)",
                 fontsize=14, fontweight="bold")
    ax.legend(fontsize=11, loc="upper right")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["B", "A"], fontsize=18, fontweight="bold")
    fig.tight_layout()
    return fig      

In [ ]:

FIXED_PRESSURE = 101.325  # kPa
def gradio_predict(smiles1, smiles2, x1):

    pressure = FIXED_PRESSURE
    if not smiles1 or not smiles2:
        return (
            """<div class="error-box">
            Please enter valid SMILES for both components.
            </div>""",
            None
        )
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)

    if mol1 is None or mol2 is None:
        return (
            """<div class="error-box">
            Error: Invalid SMILES input.
            </div>""",
            None
        )
    try:
        raw_pred = run_model(
            build_isobaric_input(
                smiles1,
                smiles2,
                pressure,
                x1
            )
        )
    except Exception as e:
        return (
            f"""<div class="error-box">
            Model Inference Error: {e}
            </div>""",
            None
        )

    pred_y1, pred_T, _ = parse_vle_output(
        raw_pred,
        "isobaric"
    )
    if pred_y1 is not None and pred_T is not None:

        result_html = f"""
        <div class="result-table">

            <div class="result-row">

                <div class="result-icon steam-icon">
                    ♨
                </div>

                <div class="result-label">
                    Vapor-phase mole fraction<br>
                    of component A
                </div>

                <div class="result-value">
                    {pred_y1:.3f}
                </div>

            </div>


            <div class="result-row">

                <div class="result-icon temp-icon">
                    &#127777;  
                </div>

                <div class="result-label">
                    Equilibrium Temperature
                </div>

                <div class="result-value">
                    {pred_T:.2f} K
                </div>

            </div>

        </div>
        """

    else:

        result_html = f"""
        <div class="error-box">
            {raw_pred}
        </div>
        """
    try:
        img = draw_phase_diagram(
            smiles1,
            smiles2,
            pressure,
            n_points=20
        )
    except Exception:
        img = None

    return result_html, img

In [ ]:
custom_css = """
/* ============================================================
   Overall Page
   ============================================================ */
body {
    background: #f7f9fc !important;
}
.gradio-container {
    max-width: 1100px !important;
    margin: 0 auto !important;
}ji'l'h
/* ============================================================
   Main Application Container
   ============================================================ */
.main-container {
    max-width: 1050px !important;
    margin: 10px auto !important;
    padding: 14px 16px 12px 16px !important;
    background: white !important;
    border: 3px solid #d9e0ea !important;
    border-radius: 20px !important;
    box-shadow:
        0px 3px 12px rgba(0,0,0,0.08) !important;
}
/* ============================================================
   Main Title
   ============================================================ */
.main-title {
    text-align: center;
    font-size: 25px;
    font-weight: 800;
    color: #111827;
    background: #eaf3ff;
    border: 1px solid #cfe2fa;

    border-radius: 10px;

    padding: 13px 10px;

    margin-bottom: 10px;
}
/* ============================================================
   Main Two-Column Layout
   ============================================================ */
.main-container > .gr-row {
    gap: 14px !important;
}
/* ============================================================
   COMPLETE SECTION BOX
   Header + Content = ONE continuous box
   ============================================================ */
.section-box {
    padding: 0 !important;
    margin: 0 0 10px 0 !important;
    gap: 0 !important;
    border: 1.5px solid #cfe0f5 !important;
    border-radius: 7px !important;
    background: white !important;
    overflow: hidden !important;
}
/* ============================================================
   Section Header
   ============================================================ */
.section-header {
    display: flex;
    align-items: center;
    min-height: 46px;
    padding: 0 12px;
    margin: 0;
    background: #f4f9ff;
    border: none;
    border-bottom: 1.5px solid #cfe0f5;
    font-size: 18px;
    font-weight: 700;
    color: #1769c2;
}
/* ============================================================
   Section Number
   ============================================================ */
.section-number {
    width: 27px;
    height: 27px;
    min-width: 27px;
    display: inline-flex;
    justify-content: center;
    align-items: center;
    margin-right: 10px;
    border-radius: 50%;
    background: #1769c2;
    color: white;
    font-size: 15px;
    font-weight: 800;
}

/* ============================================================
   Input Panel
   ============================================================ */
.input-panel {
    padding: 10px !important;
    margin: 0 !important;
    gap: 5px !important;
    background: white !important;
    border: none !important;
    border-radius: 0 !important;
}
/* ============================================================
   Gradio Input Boxes
   ============================================================ */
.input-box {
    margin: 0 0 0px 0 !important;
    padding: 0 !important;
    background-color: white !important;
}

.input-box.block,
.input-box.block.svelte-1svsvh2 {
    background: white !important;
    color-scheme: light !important;
}
.input-box.block,
.input-box.block.svelte-1svsvh2,
.input-box.block.svelte-1svsvh2.p-added.auto-margin {
    background: white !important;
    color-scheme: light !important;
}

/* Input labels */
.input-box label {
    font-size: 14px !important;
    font-weight: 600 !important;
    color: #1769c2 !important;
    margin-bottom: 7px !important; 
}
/* Input fields */
.input-box input,
.input-box textarea {
    border-radius: 7px !important;
    border: 1px solid #c7cdd5 !important;
    font-size: 14px !important;
    box-shadow: none !important;

    /* Force input fields to remain white in dark mode */
    background-color: white !important;
    color: #222 !important;
    color-scheme: light !important;
}
/* Input placeholder */
.input-box input::placeholder,
.input-box textarea::placeholder {
    color: #9aa3b2 !important;
    opacity: 1 !important;
}

/* Input field focus */
.input-box input:focus,
.input-box textarea:focus {
    border-color: #1769c2 !important;
    box-shadow:
        0 0 0 1px rgba(23,105,194,0.15) !important;
}
/* ============================================================
   Submit Button
   ============================================================ */
.predict-button {

    width: 100% !important;
    margin-top: 5px !important;
    margin-bottom: 0 !important;
    background: #1769c2 !important;
    color: white !important;
    border: none !important;
    border-radius: 7px !important;
    font-size: 15px !important;
    font-weight: 700 !important;
    padding: 9px 15px !important;
    min-height: 40px !important;
}
.predict-button:hover {
    background: #1257a2 !important;
}
/* ============================================================
   OUTPUT SECTION
   ============================================================ */

.output-section {
    margin-top: 0px !important;
    margin-bottom: 0 !important;
    border-color: #c7dfc7 !important;
}

/* ============================================================
   Output Header
   ============================================================ */
.output-header {
    display: flex;
    align-items: center;
    min-height: 45px;
    padding: 0 12px;
    background: #f3fbf3;
    border: none;
    border-bottom: 1.5px solid #c7dfc7;
    color: #32833b;
    font-size: 17px;
    font-weight: 700;
}
/* ============================================================
   Output Number
   ============================================================ */
.output-number {
    width: 27px;
    height: 27px;
    min-width: 27px;
    display: inline-flex;
    justify-content: center;
    align-items: center;
    margin-right: 10px;
    border-radius: 50%;
    background: #32833b;
    color: white;
    font-size: 15px;
    font-weight: 800;
}
/* ============================================================
   Result Table
   ============================================================ */
.result-table {
    padding: 0px 7px 7px 7px !important;
    margin: 0;
    background: white;
    border: none;
}
/* ============================================================
   Result Row
   ============================================================ */
.result-row {
    display: grid;
    grid-template-columns: 60px 1fr 120px;
    min-height: 61px;
    margin-bottom: 5px;
    border: 1px solid #c7dfc7;
    background: #f8fcf8;
    border-radius: 7px;
    overflow: hidden;
}
.result-row:last-child {
    margin-bottom: 0;
}
/* ============================================================
   Result Icon
   ============================================================ */
.result-icon {
    display: flex;
    justify-content: center;
    align-items: center;
    font-size: 28px;
    border-right: 1px solid #c7dfc7;
}
.steam-icon {
    color: #16a34a;
}
.temp-icon {
    color: #f97316;
}
/* ============================================================
   Result Label
   ============================================================ */
.result-label {
    display: flex;
    align-items: center;
    padding: 6px 10px;
    font-size: 14px;
    line-height: 1.25;
    color: #222;
}
/* ============================================================
   Result Value
   ============================================================ */
.result-value {
    display: flex;
    justify-content: center;
    align-items: center;
    background: #f1f8f1;
    border-left: 1px solid #c7dfc7;
    font-size: 16px;
    font-weight: 800;
    color: #222;
}
/* ============================================================
   PHASE DIAGRAM SECTION
   ============================================================ */
.plot-section {
    padding: 0 !important;
    margin: 0 0 10px 0 !important;
    gap: 0 !important;
    border: 1.5px solid #cfe0f5 !important;
    border-radius: 7px !important;
    background: white !important;
    overflow: hidden !important;
}
/* ============================================================
   Plot Header
   ============================================================ */
.plot-header {
    display: flex;
    align-items: center;
    min-height: 46px;
    padding: 0 12px;
    background: #f4f9ff;
    border-bottom: 1.5px solid #cfe0f5;
    color: #1769c2;
    font-size: 18px;
    font-weight: 700;
}
/* ============================================================
   Plot Number
   ============================================================ */
.plot-number {
    width: 27px;
    height: 27px;
    min-width: 27px;
    display: inline-flex;
    justify-content: center;
    align-items: center;
    margin-right: 10px;
    border-radius: 50%;
    background: #1769c2;
    color: white;
    font-size: 15px;
    font-weight: 800;
}
/* ============================================================
   Plot Container
   ============================================================ */
.plot-container {
    padding: 6px !important;
    margin: 0 !important;
    min-height: 245px !important;
    background: white !important;
    border: none !important;
    border-radius: 0 !important;
    gap: 0 !important;
}
/* ============================================================
   Application Box
   ============================================================ */
.application-box {
    display: flex;
    align-items: flex-start;
    gap: 12px;
    margin-top: 8px;
    padding: 11px 13px;
    background: #fffaf0;
    border: 1.5px solid #f3dfae;
    border-radius: 8px;
}
/* ============================================================
   Application Icon
   ============================================================ */
.application-icon {
    font-size: 30px;
    line-height: 1;
    padding-top: 2px;
    flex-shrink: 0;
}
/* ============================================================
   Application Title
   ============================================================ */
.application-title {
    font-size: 17px;
    font-weight: 800;
    margin-bottom: 3px;
    color: #222;
}
/* ============================================================
   Application Text
   ============================================================ */
.application-text {
    font-size: 14px;
    line-height: 1.45;
    color: #222;
}
/* ============================================================
   Error Box
   ============================================================ */
.error-box {
    margin: 7px;
    padding: 12px;
    background: #fff1f1;
    border: 1px solid #e5aaaa;
    border-radius: 7px;
    color: #b42318;
    font-size: 14px;
}
/* ============================================================
   Hide Unnecessary Gradio Elements
   ============================================================ */
footer {
    display: none !important;
}
/* ============================================================
   Responsive Layout
   ============================================================ */
@media (max-width: 800px) {
    .main-container {
        margin: 5px !important;
        padding: 8px !important;
    }
    .main-title {
        font-size: 19px;
    }
    .section-header,
    .plot-header {
        font-size: 15px;
    }
    .result-row {
        grid-template-columns:
            50px 1fr 105px;
    }
    .application-text {
        font-size: 13px;
    }
}

"""
# ============================================================
# Gradio Blocks
# ============================================================

with gr.Blocks(

    theme=gr.themes.Soft(

        primary_hue="blue",

        secondary_hue="green"

    ),

    css=custom_css

) as demo:


    # ========================================================
    # Main Container
    # ========================================================

    with gr.Column(
        elem_classes=["main-container"]
    ):
        # ====================================================
        # Main Title
        # ====================================================
        gr.HTML("""
        <div class="main-title">
            VLE Behavior Prediction AI (Binary Mixture)
        </div>
        """)
        # ====================================================
        # Main Two-Column Layout
        # ====================================================
        with gr.Row(
            equal_height=False
        ):
            # =================================================
            # LEFT COLUMN
            # =================================================
            with gr.Column(
                scale=1
            ):
                # =============================================
                #  INPUT SECTION
                # =============================================

                with gr.Column(
                    elem_classes=["section-box"]
                ):


                    # ---------- Header ----------

                    gr.HTML("""
                    <div class="section-header">

                        <span class="section-number">
                            1
                        </span>

                        Input : SMILES and Operating Conditions

                    </div>
                    """)


                    # ---------- Input Content ----------

                    with gr.Column(
                        elem_classes=["input-panel"]
                    ):


                        # Component A

                        comp1_smiles = gr.Textbox(

                            label="Component A (Input SMILES)",
                            value="CC(C)O",  
                            placeholder="e.g., CC(C)O",
                            
                            lines=1.1,

                            elem_classes=["input-box"]

                        )


                        # Component B

                        comp2_smiles = gr.Textbox(

                            label="Component B (Input SMILES)",
                            value="CC1=CC=CC=C1",   
                            placeholder="e.g., CC1=CC=CC=C1",

                            lines=1.1,

                            elem_classes=["input-box"]

                        )


                        # Liquid composition

                        x1 = gr.Number(

                            label="Liquid-phase mole fraction of component A",
                            value=0.500,
                            precision=3,
                            elem_classes=["input-box"]
                        )
                        # Submit Button
                        submit_btn = gr.Button(
                            "Submit and Predict",
                            elem_classes=["predict-button"]
                        )
                    # =============================================
                    # APPLICATION
                    # =============================================
                    gr.HTML("""
                    <div class="application-box">
                        <div class="application-icon">
                            💡
                        </div>
                        <div>
                            <div class="application-title">
                                Application:
                            </div>
                            <div class="application-text">
                                Given the composition and pressure (101.325 kPa),
                                the model predicts the equilibrium
                                temperature and vapor composition,
                                and can further predict VLE behavior
                                across the full composition range.
                            </div>
                        </div>
                    </div>
                    """)
                

            # =================================================
            # RIGHT COLUMN
            # =================================================
            with gr.Column(
                scale=1
            ):
                # =============================================
                #  OUTPUT SECTION
                # =============================================
                with gr.Column(
                    elem_classes=[
                        "section-box",
                        "output-section"
                    ]
                ):
                    # ---------- Header ----------
                    gr.HTML("""
                    <div class="output-header">
                        <span class="output-number">
                            2
                        </span>
                        Output : VLE Prediction Results
                    </div>
                    """)
                    # ---------- Results ----------
                    model_output = gr.HTML(

                        """
                        <div class="result-table">
                            <!-- Vapor composition -->
                            <div class="result-row">
                                <div class="result-icon steam-icon">
                                    ♨
                                </div>
                                <div class="result-label">
                                    Vapor-phase mole fraction<br>
                                    of component A
                                </div>
                                <div class="result-value">
                                    —
                                </div>
                            </div>
                            <!-- Temperature -->
                            <div class="result-row">
                                <div class="result-icon temp-icon">
                                    &#127777;  
                                </div>
                                <div class="result-label">
                                    Equilibrium Temperature
                                </div>
                                <div class="result-value">
                                    —
                                </div>
                            </div>
                        </div>
                        """
                    )
                # =============================================
                #  PHASE DIAGRAM SECTION
                # =============================================
                with gr.Column(
                    elem_classes=["plot-section"]
                ):
                    # ---------- Header ----------
                    gr.HTML("""
                    <div class="plot-header">
                        <span class="plot-number">
                            3
                        </span>
                        Predicted T-x-y Diagram

                    </div>
                    """)
                    # ---------- Plot ----------
                    with gr.Column(
                        elem_classes=["plot-container"]
                    ):
                        phase_plot = gr.Plot(
                            show_label=False,
                            container=False
                        )
    # ========================================================
    # Prediction Event
    # ========================================================
    submit_btn.click(
        fn=gradio_predict,
        inputs=[
            comp1_smiles,
            comp2_smiles,
            x1
        ],
        outputs=[
            model_output,
            phase_plot
        ]
    )

In [ ]:
demo.close()
demo.launch(server_name="0.0.0.0")